# AssetFlow — Synthetic Data Generator

Generates realistic synthetic data for **Screen 9 (Reports & Analytics)** ML tasks:
predictive maintenance, idle-asset detection, retirement scoring, and booking heatmaps.

**Patterns baked in on purpose** so the models find real signal:
- Older assets + certain categories (Vehicles, Heavy Equipment) → more maintenance events
- Some resources are genuinely popular, others idle
- Bookings follow realistic weekday/peak-hour patterns
- A handful of assets are overdue on return

Run all cells top to bottom. Output CSVs are saved to `../assetflow_synthetic_data/`.


In [ ]:
import numpy as np
import pandas as pd
from faker import Faker
from datetime import datetime, timedelta
import random
import os

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
fake = Faker()
Faker.seed(SEED)

TODAY = datetime(2026, 7, 12)
OUT_DIR = "assetflow_synthetic_data"
os.makedirs(OUT_DIR, exist_ok=True)


## 1. Departments

In [ ]:
departments_raw = [
    ("Engineering", None, "Active"),
    ("Facilities", None, "Active"),
    ("Field Ops (East)", "Facilities", "Active"),
    ("IT", None, "Active"),
    ("HR", None, "Active"),
    ("Finance", None, "Active"),
    ("Procurement", None, "Active"),
    ("Warehouse Ops", "Facilities", "Inactive"),
]

departments = []
for i, (name, parent, status) in enumerate(departments_raw, start=1):
    departments.append({
        "department_id": f"D-{i:03d}",
        "department_name": name,
        "parent_department": parent,
        "head": fake.name(),
        "status": status,
    })
df_departments = pd.DataFrame(departments)
df_departments


## 2. Asset Categories\n\n`failure_weight` is an internal generation knob (not a real app field) that drives how often each category needs maintenance.

In [ ]:
categories_raw = [
    ("Electronics", "warranty_months", 0.35),
    ("Furniture", None, 0.05),
    ("Vehicles", "insurance_expiry", 0.55),
    ("Heavy Equipment", "service_interval_days", 0.60),
    ("IT Peripherals", "warranty_months", 0.20),
    ("HVAC/Appliances", "warranty_months", 0.40),
]
categories = []
for i, (name, field, fail_weight) in enumerate(categories_raw, start=1):
    categories.append({
        "category_id": f"C-{i:03d}",
        "category_name": name,
        "special_field": field,
        "failure_weight": fail_weight,
    })
df_categories = pd.DataFrame(categories)
df_categories


## 3. Employees

In [ ]:
N_EMPLOYEES = 60
roles = ["Employee"] * 50 + ["Department Head"] * 6 + ["Asset Manager"] * 3 + ["Admin"] * 1
random.shuffle(roles)
while len(roles) < N_EMPLOYEES:
    roles.append("Employee")
roles = roles[:N_EMPLOYEES]

employees = []
for i in range(1, N_EMPLOYEES + 1):
    dept = df_departments.sample(1).iloc[0]
    employees.append({
        "employee_id": f"E-{i:04d}",
        "name": fake.name(),
        "email": fake.company_email(),
        "department_id": dept["department_id"],
        "department_name": dept["department_name"],
        "role": roles[i - 1],
        "status": np.random.choice(["Active", "Inactive"], p=[0.93, 0.07]),
    })
df_employees = pd.DataFrame(employees)
df_employees.head()


## 4. Assets\n\nAge is exponentially distributed (mostly newer assets, a long tail of old ones) — this is what powers the retirement/maintenance signal later.

In [ ]:
N_ASSETS = 350

asset_name_pool = {
    "Electronics": ["Dell Laptop", "HP Laptop", "Projector", "Desktop PC", "Monitor", "Camera"],
    "Furniture": ["Office Chair", "Standing Desk", "Bookshelf", "Cabinet", "Meeting Table"],
    "Vehicles": ["Delivery Van", "Sedan Car", "Pickup Truck", "Scooter"],
    "Heavy Equipment": ["Forklift", "Generator", "Compressor", "Pallet Jack"],
    "IT Peripherals": ["Keyboard", "Wireless Mouse", "Docking Station", "Webcam"],
    "HVAC/Appliances": ["AC Unit", "Water Cooler", "Air Purifier", "Microwave"],
}

locations = ["Bengaluru HQ Floor 1", "Bengaluru HQ Floor 2", "Warehouse - East",
             "Warehouse - West", "Mumbai Office", "Delhi Office", "Field Site - Pune"]

statuses = ["Available", "Allocated", "Reserved", "Under Maintenance", "Lost", "Retired", "Disposed"]
status_weights = [0.30, 0.38, 0.06, 0.10, 0.02, 0.09, 0.05]

assets = []
for i in range(1, N_ASSETS + 1):
    cat = df_categories.sample(1, weights=df_categories["failure_weight"] + 0.1).iloc[0]
    name = random.choice(asset_name_pool[cat["category_name"]])

    days_old = int(np.random.exponential(scale=500))
    days_old = min(days_old, 2200)
    acquisition_date = TODAY - timedelta(days=days_old)

    cost = {
        "Electronics": np.random.uniform(25000, 120000),
        "Furniture": np.random.uniform(3000, 25000),
        "Vehicles": np.random.uniform(400000, 1500000),
        "Heavy Equipment": np.random.uniform(150000, 800000),
        "IT Peripherals": np.random.uniform(800, 6000),
        "HVAC/Appliances": np.random.uniform(15000, 90000),
    }[cat["category_name"]]

    status = np.random.choice(statuses, p=status_weights)
    is_bookable = 1 if cat["category_name"] in ["Vehicles", "Heavy Equipment"] or name in [
        "Meeting Table", "Projector"] else np.random.choice([0, 1], p=[0.85, 0.15])

    condition = np.random.choice(
        ["Excellent", "Good", "Fair", "Poor"],
        p=[0.30, 0.40, 0.22, 0.08] if days_old < 900 else [0.10, 0.35, 0.35, 0.20]
    )

    assets.append({
        "asset_tag": f"AF-{i:04d}",
        "asset_name": name,
        "category_id": cat["category_id"],
        "category_name": cat["category_name"],
        "serial_number": fake.bothify(text="SN-########"),
        "acquisition_date": acquisition_date.date().isoformat(),
        "acquisition_cost": round(cost, 2),
        "condition": condition,
        "location": random.choice(locations),
        "status": status,
        "is_bookable": is_bookable,
        "age_days": days_old,
    })
df_assets = pd.DataFrame(assets)
df_assets.head()


## 5. Allocations (allocation history)

In [ ]:
allocations = []
alloc_id = 1
for _, asset in df_assets.iterrows():
    if asset["category_name"] in ["Vehicles", "Heavy Equipment"] and asset["is_bookable"] == 1:
        continue

    n_history = np.random.poisson(1.5) + (1 if asset["status"] == "Allocated" else 0)
    n_history = min(n_history, 5)
    current_date = pd.to_datetime(asset["acquisition_date"])

    for h in range(n_history):
        emp = df_employees.sample(1).iloc[0]
        alloc_start = current_date + timedelta(days=np.random.randint(5, 90))
        if alloc_start > TODAY:
            break
        expected_return = alloc_start + timedelta(days=int(np.random.choice([30, 60, 90, 180, 365])))

        is_last = (h == n_history - 1)
        if is_last and asset["status"] == "Allocated":
            actual_return = None
        else:
            actual_return = expected_return + timedelta(days=np.random.randint(-10, 15))
            if actual_return > TODAY:
                actual_return = TODAY - timedelta(days=np.random.randint(1, 20))

        overdue = 0
        if actual_return is None and expected_return < TODAY:
            overdue = 1
        elif actual_return is not None and actual_return > expected_return:
            overdue = 1

        allocations.append({
            "allocation_id": f"AL-{alloc_id:05d}",
            "asset_tag": asset["asset_tag"],
            "employee_id": emp["employee_id"],
            "department_id": emp["department_id"],
            "department_name": emp["department_name"],
            "allocation_date": alloc_start.date().isoformat(),
            "expected_return_date": expected_return.date().isoformat(),
            "actual_return_date": actual_return.date().isoformat() if actual_return is not None else None,
            "overdue_flag": overdue,
        })
        alloc_id += 1
        current_date = alloc_start

df_allocations = pd.DataFrame(allocations)
df_allocations.head()


## 6. Maintenance Logs\n\nEvent count per asset = f(category failure-weight, asset age). This relationship is exactly what your predictive-maintenance model should learn back out.

In [ ]:
priorities = ["Low", "Medium", "High", "Critical"]
maint_statuses_pool = ["Pending", "Approved", "Rejected", "Technician Assigned", "In Progress", "Resolved"]
issues_pool = [
    "Not turning on", "Noisy compressor", "Screen flickering", "Battery not charging",
    "Overheating", "Parts worn out", "Leaking fluid", "Unusual vibration",
    "Software crash", "Physical damage", "AC not cooling", "Motor failure",
]

maintenance = []
maint_id = 1
cat_weight_map = dict(zip(df_categories["category_name"], df_categories["failure_weight"]))

for _, asset in df_assets.iterrows():
    weight = cat_weight_map[asset["category_name"]]
    age_factor = min(asset["age_days"] / 365, 5) / 5
    expected_events = weight * 3 + age_factor * 2.5
    n_events = np.random.poisson(max(expected_events, 0.05))

    acq = pd.to_datetime(asset["acquisition_date"])
    for _ in range(n_events):
        raised_date = acq + timedelta(days=np.random.randint(10, max(asset["age_days"], 15)))
        if raised_date > TODAY:
            continue
        priority = np.random.choice(priorities, p=[0.35, 0.35, 0.22, 0.08])
        wf_status = np.random.choice(maint_statuses_pool, p=[0.10, 0.12, 0.05, 0.10, 0.13, 0.50])

        resolved_date = None
        if wf_status == "Resolved":
            resolved_date = raised_date + timedelta(days=np.random.randint(1, 21))
            if resolved_date > TODAY:
                resolved_date = TODAY

        maintenance.append({
            "maintenance_id": f"M-{maint_id:05d}",
            "asset_tag": asset["asset_tag"],
            "category_name": asset["category_name"],
            "issue_description": random.choice(issues_pool),
            "priority": priority,
            "status": wf_status,
            "raised_date": raised_date.date().isoformat(),
            "resolved_date": resolved_date.date().isoformat() if resolved_date else None,
            "technician": fake.name() if wf_status in ["Technician Assigned", "In Progress", "Resolved"] else None,
        })
        maint_id += 1

df_maintenance = pd.DataFrame(maintenance)
df_maintenance.head()


## 7. Resource Bookings\n\nSome resources are popular, some idle. Weekday/peak-hour weighting bakes in a realistic office pattern for the heatmap.

In [ ]:
bookable_assets = df_assets[df_assets["is_bookable"] == 1]

hour_slots = list(range(9, 18))
peak_hours = {10, 11, 14, 15}
weekday_weight = {0: 1.0, 1: 1.1, 2: 1.15, 3: 1.1, 4: 0.9, 5: 0.2, 6: 0.1}

def hour_probs(slots, peaks, peak_boost=2.5):
    weights = np.array([peak_boost if h in peaks else 1.0 for h in slots], dtype=float)
    return weights / weights.sum()

bookings = []
booking_id = 1

for _, asset in bookable_assets.iterrows():
    popularity = np.random.choice(["high", "medium", "low"], p=[0.25, 0.45, 0.30])
    n_bookings = {"high": np.random.randint(30, 60),
                  "medium": np.random.randint(10, 30),
                  "low": np.random.randint(0, 6)}[popularity]

    for _ in range(n_bookings):
        days_back = np.random.randint(0, 365)
        booking_date = TODAY - timedelta(days=days_back)
        weekday = booking_date.weekday()
        if np.random.random() > weekday_weight.get(weekday, 0.5) / 1.15:
            continue

        hour = np.random.choice(hour_slots, p=hour_probs(hour_slots, peak_hours))
        duration = np.random.choice([30, 60, 90, 120], p=[0.3, 0.4, 0.2, 0.1])

        status = "Completed" if booking_date < TODAY else np.random.choice(["Upcoming", "Ongoing"], p=[0.85, 0.15])
        if np.random.random() < 0.08:
            status = "Cancelled"

        bookings.append({
            "booking_id": f"BK-{booking_id:05d}",
            "asset_tag": asset["asset_tag"],
            "asset_name": asset["asset_name"],
            "booking_date": booking_date.date().isoformat(),
            "weekday": booking_date.strftime("%A"),
            "start_hour": int(hour),
            "duration_minutes": int(duration),
            "booked_by": df_employees.sample(1).iloc[0]["employee_id"],
            "status": status,
        })
        booking_id += 1

df_bookings = pd.DataFrame(bookings)
df_bookings.head()


## 8. Save all tables to CSV

In [ ]:
df_departments.to_csv(f"{OUT_DIR}/departments.csv", index=False)
df_categories.drop(columns=["failure_weight"]).to_csv(f"{OUT_DIR}/categories.csv", index=False)
df_employees.to_csv(f"{OUT_DIR}/employees.csv", index=False)
df_assets.drop(columns=["age_days"]).to_csv(f"{OUT_DIR}/assets.csv", index=False)
df_allocations.to_csv(f"{OUT_DIR}/allocations.csv", index=False)
df_maintenance.to_csv(f"{OUT_DIR}/maintenance.csv", index=False)
df_bookings.to_csv(f"{OUT_DIR}/bookings.csv", index=False)

print("Departments:", df_departments.shape)
print("Categories:", df_categories.shape)
print("Employees:", df_employees.shape)
print("Assets:", df_assets.shape)
print("Allocations:", df_allocations.shape)
print("Maintenance:", df_maintenance.shape)
print("Bookings:", df_bookings.shape)
print("\nSaved to:", OUT_DIR)
